In [3]:
import matplotlib.pyplot as plt
def lzw_encode(data: bytes, max_dict_size: int = 4096):
    if not data:
        return b''
    
    encoded = bytearray()
    n = len(data)
    pos = 0
    
    # Инициализируем словарь всеми возможными байтами
    dictionary = {bytes([i]): i for i in range(256)}
    next_index = 256
    
    current_phrase = b""
    
    while pos < n:
        # Добавляем следующий байт
        current_byte = bytes([data[pos]])
        new_phrase = current_phrase + current_byte
        
        if new_phrase in dictionary:
            if next_index < max_dict_size:
                # Фраза есть в словаре, продолжаем
                current_phrase = new_phrase
                pos += 1
            else:
                # Словарь полон, но фраза существует.
                # Выдаем код для текущей фразы и начинаем новую с текущего байта.
                if current_phrase:  # Защита от пустой строки
                    prefix_index = dictionary[current_phrase]
                    encoded.extend(prefix_index.to_bytes(2, 'big'))
                current_phrase = current_byte
                pos += 1
        else:
            # Выдаем индекс текущей фразы
            prefix_index = dictionary[current_phrase]
            encoded.extend(prefix_index.to_bytes(2, 'big'))
            
            # Добавляем новую фразу в словарь (если не превышен лимит)
            if new_phrase and next_index < max_dict_size:
                dictionary[new_phrase] = next_index
                next_index += 1
            
            # Сбрасываем текущую фразу
            current_phrase = current_byte
            pos += 1
    
    # Обработка остатка
    if current_phrase:
        prefix_index = dictionary[current_phrase]
        encoded.extend(prefix_index.to_bytes(2, 'big'))
    
    return bytes(encoded)


def lzw_decode(encoded: bytes, max_dict_size: int = 4096):
    if not encoded:
        return b''
    
    decoded = bytearray()
    n = len(encoded)
    i = 0
    
    # Инициализируем словарь всеми возможными байтами
    dictionary = {i: bytes([i]) for i in range(256)}
    next_index = 256
    
    # Читаем первый индекс
    if i + 2 > n:
        return b''
    
    prev_index = int.from_bytes(encoded[i:i+2], 'big')
    i += 2
    
    # Первая фраза
    prev_phrase = dictionary.get(prev_index, b"")
    decoded.extend(prev_phrase)
    
    while i + 2 <= n:
        # Читаем следующий индекс
        curr_index = int.from_bytes(encoded[i:i+2], 'big')
        i += 2
        
        # Получаем фразу
        if curr_index in dictionary:
            curr_phrase = dictionary[curr_index]
        elif curr_index == next_index and next_index < max_dict_size:
            # Специальный случай: фраза, которую только будем добавлять
            curr_phrase = prev_phrase + bytes([prev_phrase[0]])
        else:
            # Некорректный индекс
            break
        
        # Добавляем в декодированный вывод
        decoded.extend(curr_phrase)
        
        # Добавляем новую фразу в словарь
        if next_index < max_dict_size and prev_phrase:
            new_phrase = prev_phrase + bytes([curr_phrase[0]])
            dictionary[next_index] = new_phrase
            next_index += 1
        
        prev_phrase = curr_phrase
    
    return bytes(decoded)

def compress(input_path: str, output_path: str, max_dict_size: int = 4096):
    with open(input_path, 'rb') as f:
        original_data = f.read()
    
    compressed_data = lzw_encode(original_data, max_dict_size)
    
    with open(output_path, 'wb') as f:
        f.write(len(original_data).to_bytes(4, 'big'))
        f.write(compressed_data)

def decompress(input_path: str, output_path: str):
        with open(input_path, 'rb') as f:
            original_size_data = f.read(4)
            compressed_data = f.read()
        
        decompressed_data = lzw_decode(compressed_data)

        with open(output_path, 'wb') as f:
            f.write(decompressed_data)

TEST_FILES = [
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/enwik7",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test2.txt",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test3.exe",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test4.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test5.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test6.raw",
]

TEST_FILES_COMPRESS = [
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw/enwik7_comp",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw/test2_comp.txt",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw/test3_comp.exe",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw/test4_comp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw/test5_comp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw/test6_comp.raw",
]

TEST_FILES_DECOMPRESS = [
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw/enwik7_decomp",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw/test2_decomp.txt",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw/test3_decomp.exe",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw/test4_decomp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw/test5_decomp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzw/test6_decomp.raw",
]

from pathlib import Path

for file_path in range(len(TEST_FILES)):
    
    original_size = Path(TEST_FILES[file_path]).stat().st_size
    
    print(f"\nФайл: {file_path+1} ({original_size:,} байт)")

    # Сжатие
    compress(TEST_FILES[file_path], TEST_FILES_COMPRESS[file_path])
    compressed_size = Path(TEST_FILES_COMPRESS[file_path]).stat().st_size
    
    # Распаковка
    decompress(TEST_FILES_COMPRESS[file_path], TEST_FILES_DECOMPRESS[file_path])
    decompressed_size = Path(TEST_FILES_DECOMPRESS[file_path]).stat().st_size
    
    # Проверка целостности
    if decompressed_size == original_size:
        print("correct")
    else:
        print("fail")
    
    # Статистика
    ratio = (original_size / compressed_size) if compressed_size > 0 else 0
    print(f"Исходный размер: {original_size}\n")
    print(f"После сжатия: {compressed_size}\n")
    print(f"После распаковки: {decompressed_size}\n")
    print(f"Коэффициент сжатия: {ratio}\n")


Файл: 1 (10,000,000 байт)
correct
Исходный размер: 10000000

После сжатия: 19987636

После распаковки: 10000000

Коэффициент сжатия: 0.5003092912038222


Файл: 2 (371,816 байт)
correct
Исходный размер: 371816

После сжатия: 722276

После распаковки: 371816

Коэффициент сжатия: 0.5147838222507739


Файл: 3 (1,104,440 байт)
correct
Исходный размер: 1104440

После сжатия: 2188966

После распаковки: 1104440

Коэффициент сжатия: 0.5045487230043774


Файл: 4 (1,532,562 байт)
correct
Исходный размер: 1532562

После сжатия: 3050038

После распаковки: 1532562

Коэффициент сжатия: 0.502473083941905


Файл: 5 (2,001,012 байт)
correct
Исходный размер: 2001012

После сжатия: 3983236

После распаковки: 2001012

Коэффициент сжатия: 0.5023583839872907


Файл: 6 (1,806,348 байт)
correct
Исходный размер: 1806348

После сжатия: 3611638

После распаковки: 1806348

Коэффициент сжатия: 0.500146470936456

